# 05 — LLM Agent (Evidence-Traced Graph QA)

**Pipeline Step 5 of 5**

This notebook demonstrates the LLM-powered QA agent that queries the Micro-CKG with strict evidence traceability. Every answer cites exact `(Source)--[Edge_Type, Score=X.XX]-->(Target)` graph evidence.

### Hardened Guardrails
1. **No External Knowledge** — answers from graph context only
2. **Missing Data Fallback** — explicit "No evidence found" response
3. **Mandatory Citation** — `[Evidence: (Source) --(Edge)--> (Target)]`
4. **Objective Tone** — no speculation

### Prerequisites
- Ollama running locally with `deepseek-r1:14b` pulled

### Inputs
| File | Description |
|---|---|
| `cache/micro_ckg.graphml` | Serialized Micro-CKG from Step 04 |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from src.biocypher_adapter import load_graph
from src.llm_agent import create_qa_agent, query_graph

CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 5.1 Load Micro-CKG

We load the persisted GraphML file produced by notebook 04. The graph is deserialized back into a NetworkX DiGraph with all node and edge attributes intact. The summary below confirms the graph structure matches expectations (number of gene nodes, cell-type nodes, region nodes, and total edges).

In [2]:
graph = load_graph(CACHE_DIR / "micro_ckg.graphml")
print(f"\nMicro-CKG: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

  Graph loaded: 57 nodes, 483 edges

Micro-CKG: 57 nodes, 483 edges


## 5.2 Create QA Agent (Ollama — deepseek-r1:14b)

The QA agent uses **deepseek-r1:14b** running locally via Ollama with a pruned graph context. Weak edges (|log2FC| < 0.25 and spatial correlation < 0.4) are removed to reduce prompt token size. No API key needed — runs entirely on-device.

In [3]:
import time

# Aggressively prune the graph to reduce LLM prompt token size
optimized_graph = graph.copy()
edges_to_remove = [(u, v) for u, v, d in optimized_graph.edges(data=True)
                   if abs(d.get("log2fc", 0)) < 0.25 and d.get("spatial_correlation", 0) < 0.4]
optimized_graph.remove_edges_from(edges_to_remove)

# Strip internal-only attributes so they don't appear in the LLM context
_INTERNAL_ATTRS = {"human_ortholog"}
for _, data in optimized_graph.nodes(data=True):
    for attr in _INTERNAL_ATTRS:
        data.pop(attr, None)

print(f"Original edges: {graph.number_of_edges()}")
print(f"Pruned edges for LLM: {optimized_graph.number_of_edges()}")

agent = create_qa_agent(optimized_graph, provider="ollama")
print("QA agent initialised (Ollama — deepseek-r1:14b).")

Original edges: 483
Pruned edges for LLM: 400


QA agent initialised (Ollama — deepseek-r1:14b).


## 5.3 Killer Demo Queries

Three questions answered by the local deepseek-r1:14b model:

1. **Biomarker Discovery** — What statistically significant genes drive the AD condition?
2. **Spatial Context** — Which anatomical regions harbor key neuroinflammatory markers?
3. **Anti-Hallucination Trap** — Ask about Tau tangles and Lecanemab (neither in graph).

In [ ]:
print("[Query 1] Discovering AD drivers...")
print(query_graph(agent, "Based on the Micro-CKG, what are the top statistically significant genes driving the AD condition? Refer to all genes by their mouse gene symbol only."))

print("\n[Query 2] Finding spatial context...")
print(query_graph(agent, "Which specific spatial clusters or anatomical regions are Fth1 and Prnp mapped to?"))

print("\n[Query 3] Anti-Hallucination check...")
print(query_graph(agent, "Does the graph indicate any relationship between Tau tangle formation and the FDA-approved drug Lecanemab?"))

[Query 1] Discovering AD drivers...
  Querying agent: Based on the Micro-CKG, what are the top statistically significant genes driving...


Based on the Micro-CKG data provided, the top statistically significant genes driving Alzheimer's Disease (AD) are:

1. **Tsc22d4**: This gene consistently appears with very low adjusted p-values (pval_adj=0.0) across multiple clusters, indicating strong statistical significance. It is associated with various brain regions, including cortex, cerebellum, hippocampus, thalamus, and white matter.

2. **Trf**: Similarly, Trf shows significant associations with low p-values across multiple clusters, including cortex, cerebellum, hippocampus, thalamus, and white matter. It also has a notable log2fc of 3.9478 in Cluster_9_White_Matter, indicating substantial upregulation in AD.

3. **TSC22D4**: This appears to be the same gene as Tsc22d4, given the similarity in name and data patterns. It is consistently significant across various clusters.

These genes are highlighted due to their consistent low p-values and significant log2fc values, suggesting their pivotal role in driving AD as per the Mi

Based on the data provided, Fth1 and Prnp are mapped to the following specific spatial clusters and anatomical regions:

- **Fth1**:
  - **Spatial Clusters**: Cluster 0 (Cortex), Cluster 10 (White Matter), Cluster 12 (White Matter)
  - **Anatomical Regions**: Thalamus, White Matter

- **Prnp**:
  - **Spatial Clusters**: Cluster 14 (White Matter), Cluster 17 (White Matter)
  - **Anatomical Regions**: White Matter

These mappings indicate the specific areas where each gene is expressed.

[Query 3] Anti-Hallucination check...
  Querying agent: Does the graph indicate any relationship between Tau tangle formation and the FD...


The graph does not indicate any direct or indirect relationship between Tau tangle formation and the FDA-approved drug Lecanemab. Lecanemab is connected to amyloid-beta-related genes and other drugs, but there are no links to TAU or Tau tangles in the provided data. Therefore, based on the graph, there is no evidence suggesting a connection between Lecanemab and Tau tangle formation.
